# Evaluate Models — NEW-with-placeholder-change vs OLD

Compares **`model_new_with_change`** (denominator fix + experian placeholder
change, from `Build_Model_New_With_Change.ipynb`) against **`model_old`**
(shipping behavior). `model_new` (the original experiment's fix-only model) is
loaded alongside as a reference point, so you can see whether any movement
comes from the placeholder change specifically or was already there.

Only experian's trade features differ between `new_with_change` and `new`,
so the experian-only slices are the ones to watch.

Same eval frame as `Evaluate_Models.ipynb`: test samples' app + target,
thin-file flag, a probe percent feature compared across variants, then AUC
overall / per bureau / thin-file / changed-rows. Run in the model-engine
kernel after the build finishes.

In [2]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
sys.path.insert(0, os.getcwd())
from configs import DATA_DIR

TARGET     = 'final_DQ60_m24'
BUREAUS    = ['equifax', 'experian', 'transunion']
MODELS_DIR = os.path.join(DATA_DIR, 'models')
NEW_DIR    = os.path.join(DATA_DIR, 'new_normalized_and_processed')

# variant -> where each bureau's TEST trade features live
def trade_dir(variant, bureau):
    if variant == 'new_with_change' and bureau == 'experian':
        return os.path.join(NEW_DIR, 'experian_test', 'processed')
    v = 'new' if variant == 'new_with_change' else variant
    return os.path.join(DATA_DIR, 'samples', f'{bureau}_test', f'processed_{v}')

pd.set_option('display.float_format', lambda x: f'{x:.5f}')
print('models dir:', MODELS_DIR)

models dir: /home/jag/payment-processor-research/payment_processing_research_data/models


In [3]:
DATA_DIR

'/home/jag/payment-processor-research/payment_processing_research_data'

In [6]:
# base frame: app (tagged with bureau) + target, test samples
app = pd.concat([
    pd.read_parquet(os.path.join(DATA_DIR, 'samples', f'{b}_test', 'app.parquet'),
                    columns=['ZEST_KEY', 'appDate']).assign(bureau=b)
    for b in BUREAUS], ignore_index=True)
tgt = pd.concat([
    pd.read_parquet(os.path.join(DATA_DIR, 'samples', f'{b}_test', 'target.parquet'),
                    columns=['ZEST_KEY', TARGET])
    for b in BUREAUS], ignore_index=True)
base = app.merge(tgt, on='ZEST_KEY', how='inner')
print('base:', base.shape)
print(base['bureau'].value_counts())

base: (1200000, 4)
bureau
equifax       400000
experian      400000
transunion    400000
Name: count, dtype: int64


In [7]:
# thin-file flag from the two NON-percent trade cols (date/count based --
# unaffected by the payment-pattern changes, so read once from 'new')
TRADE_COLS = ['trade_months_since_oldest_account_opened__all_accounts',
              'trade_count__all_accounts']

def load_trade_cols(variant, cols):
    parts = []
    for b in BUREAUS:
        df = pd.read_parquet(trade_dir(variant, b), columns=cols)
        parts.append(df.reset_index())   # ZEST_KEY index -> column
    return pd.concat(parts, ignore_index=True)

base = base.merge(load_trade_cols('new', TRADE_COLS), on='ZEST_KEY', how='left')
base['flg_thin_file'] = ((base[TRADE_COLS[0]] <= 6) | (base[TRADE_COLS[1]] <= 2))
print('thin file rate:', base['flg_thin_file'].mean().round(4))

thin file rate: 0.0551


In [8]:
# probe percent feature: where do new_with_change and old actually differ?
PCT_COL = 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts'

# idempotent: drop the probe cols if this cell already ran, so the merges
# below never create _x/_y duplicates on a re-run
base = base.drop(columns=[PCT_COL + '__wc', PCT_COL + '__old', 'file_changed'], errors='ignore')

pct_wc  = load_trade_cols('new_with_change', [PCT_COL]).rename(columns={PCT_COL: PCT_COL + '__wc'})
pct_old = load_trade_cols('old',             [PCT_COL]).rename(columns={PCT_COL: PCT_COL + '__old'})
base = (base.merge(pct_wc,  on='ZEST_KEY', how='left')
            .merge(pct_old, on='ZEST_KEY', how='left'))

mask_missing = base[PCT_COL + '__wc'].isna() | base[PCT_COL + '__old'].isna()
# float (1.0/0.0/NaN), not bool: assigning NaN into a bool column makes it
# object dtype and .mean().round() breaks on object
base['file_changed'] = (base[PCT_COL + '__wc'] != base[PCT_COL + '__old']).astype('float64')
base.loc[mask_missing, 'file_changed'] = np.nan

print('rows where the probe feature changed (new_with_change vs old):')
print(base.groupby('bureau')['file_changed'].mean().round(4))
print('overall:', round(base['file_changed'].mean(), 4))

rows where the probe feature changed (new_with_change vs old):
bureau
equifax      0.07010
experian     0.06420
transunion   0.06590
Name: file_changed, dtype: float64
overall: 0.0668


In [9]:
# merge in the three score sets
VARIANTS = ['new_with_change', 'new', 'old']
SCORE_COLS = {}

def load_scores(variant):
    p = os.path.join(MODELS_DIR, f'model_{variant}', 'test_scores.parquet')
    s = pd.read_parquet(p)
    if 'ZEST_KEY' not in s.columns:
        s = s.reset_index()
    assert 'ZEST_KEY' in s.columns, f'no ZEST_KEY in {p} (cols={list(s.columns)})'
    raw_cols = [c for c in s.columns if c != 'ZEST_KEY']
    s = s.rename(columns={c: f'{c}_{variant}' for c in raw_cols})
    suffixed = [f'{c}_{variant}' for c in raw_cols]
    pred = next((c for c in suffixed if any(k in c.lower() for k in ('score', 'pred', 'prob'))),
                suffixed[0])
    SCORE_COLS[variant] = pred
    print(f'{variant}: {len(s):,} rows | prediction -> {pred}')
    return s[['ZEST_KEY', pred]]

scored = base.copy()
for v in VARIANTS:
    scored = scored.merge(load_scores(v), on='ZEST_KEY', how='left')
print('\nscored:', scored.shape)

new_with_change: 1,200,000 rows | prediction -> final_model_predictions_new_with_change
new: 1,200,000 rows | prediction -> final_model_predictions_new
old: 1,200,000 rows | prediction -> final_model_predictions_old

scored: (1200000, 13)


In [10]:
# AUC: every variant on every slice. new_with_change vs old is the headline;
# new is the reference that isolates the placeholder change.
def auc_table(df, label):
    y = df[TARGET]
    row = {'slice': label, 'n': int(len(df)), 'bad_rate': float(y.mean())}
    for v in VARIANTS:
        col = SCORE_COLS[v]
        m = y.notna() & df[col].notna()
        row[f'auc_{v}'] = roc_auc_score(y[m], df.loc[m, col]) if m.sum() and y[m].nunique() > 1 else np.nan
    return row

rows = [auc_table(scored, 'overall')]
for b in BUREAUS:
    rows.append(auc_table(scored[scored['bureau'] == b], b))
rows.append(auc_table(scored[scored['flg_thin_file'] == True], 'thin_file'))
rows.append(auc_table(scored[scored['file_changed'] == True],  'probe_feature_changed'))
rows.append(auc_table(scored[(scored['bureau'] == 'experian') & (scored['file_changed'] == True)],
                      'experian_and_changed'))

auc = pd.DataFrame(rows).set_index('slice')
auc['wc_minus_old'] = auc['auc_new_with_change'] - auc['auc_old']
auc['wc_minus_new'] = auc['auc_new_with_change'] - auc['auc_new']
auc.round(4)

,n,bad_rate,auc_new_with_change,auc_new,auc_old,wc_minus_old,wc_minus_new
slice,,,,,,,
overall,1200000,0.07100,0.77630,0.77630,0.77640,-0.00010,-0.00000
equifax,400000,0.06980,0.78020,0.78020,0.78020,0.00000,0.00000
experian,400000,0.07810,0.77820,0.77820,0.77830,-0.00010,-0.00000
transunion,400000,0.06490,0.77170,0.77180,0.77170,0.00010,-0.00000
thin_file,66089,0.14320,0.67540,0.67600,0.67650,-0.00110,-0.00060
probe_feature_changed,79028,0.15410,0.69210,0.69230,0.69250,-0.00040,-0.00020
experian_and_changed,25341,0.18190,0.69840,0.69830,0.69910,-0.00070,0.00010


In [11]:
# score-level agreement: how much did the placeholder change move scores?
from scipy.stats import spearmanr

for a, b_ in [('new_with_change', 'old'), ('new_with_change', 'new')]:
    ca, cb = SCORE_COLS[a], SCORE_COLS[b_]
    for slc, df in [('overall', scored), ('experian', scored[scored['bureau'] == 'experian'])]:
        m = df[ca].notna() & df[cb].notna()
        rho = spearmanr(df.loc[m, ca], df.loc[m, cb]).statistic
        mad = (df.loc[m, ca] - df.loc[m, cb]).abs().mean()
        print(f'{a} vs {b_:<4} [{slc:9}]: spearman={rho:.5f}  mean|diff|={mad:.6f}  n={m.sum():,}')

new_with_change vs old  [overall  ]: spearman=0.99821  mean|diff|=0.005298  n=1,200,000
new_with_change vs old  [experian ]: spearman=0.99808  mean|diff|=0.005555  n=400,000
new_with_change vs new  [overall  ]: spearman=0.99901  mean|diff|=0.004028  n=1,200,000
new_with_change vs new  [experian ]: spearman=0.99883  mean|diff|=0.004337  n=400,000


In [12]:
# feature-level drift on the model's actual FE matrix:
# new_with_change vs old, and new_with_change vs new (placeholder change only)
def load_fe(variant):
    return pd.read_parquet(os.path.join(MODELS_DIR, f'model_{variant}', 'test_fe_data.parquet'))

fe_wc, fe_old, fe_new = load_fe('new_with_change'), load_fe('old'), load_fe('new')

def diff_table(a, b, label, top=15):
    cols = [c for c in a.columns if c in b.columns]
    idx  = a.index.intersection(b.index)
    a, b = a.loc[idx, cols], b.loc[idx, cols]
    rows = []
    for c in cols:
        if a[c].dtype.kind not in 'fiub':
            continue
        af, bf = a[c].astype('float64'), b[c].astype('float64')
        neq = ~np.isclose(af, bf, equal_nan=True)
        if neq.any():
            rows.append({'col': c, 'n_changed': int(neq.sum()),
                         'pct_rows': 100 * neq.mean(),
                         'mean_abs_diff': float((af - bf).abs().mean())})
    d = pd.DataFrame(rows).sort_values('n_changed', ascending=False)
    print(f'\n=== {label}: {len(d)} of {len(cols)} numeric columns differ ===')
    if len(d):
        print(d.head(top).to_string(index=False))
    return d

diff_wc_old = diff_table(fe_wc, fe_old, 'new_with_change vs old')
diff_wc_new = diff_table(fe_wc, fe_new, 'new_with_change vs new (placeholder change only)')


=== new_with_change vs old: 1143 of 1144 numeric columns differ ===
                                                                            col  n_changed  pct_rows  mean_abs_diff
                trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts      79028   6.58567        0.00133
             trade_mean_percent_of_DQ30_in_last_24_months__active_open_accounts      77897   6.49142        0.00130
                 trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts      70224   5.85200        0.00439
              trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts      69315   5.77625        0.00427
          trade_mean_percent_of_DQ30_in_last_24_months__non_derog_open_accounts      68458   5.70483        0.00112
         trade_mean_percent_of_DQ30_in_last_24_months__individual_open_accounts      65739   5.47825        0.00126
           trade_max_percent_of_DQ30_in_last_24_months__non_derog_open_accounts      61630   5.13583        0.00367
tra